In [6]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns",50)
pd.set_option("display.width", 140)

INTERIM_DIR = os.path.join("..", "data", "interim")
PROCESSED_DIR = os.path.join("..", "data", "proccesed")
REJECTED_DIR = os.path.join("..", "data", "rejects")
os.makedirs(REJECTED_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)


routes = pd.read_pickle(os.path.join(INTERIM_DIR,"routes.pkl"))
vehicles = pd.read_pickle(os.path.join(INTERIM_DIR,"vehicles.pkl"))
stops = pd.read_pickle(os.path.join(INTERIM_DIR,"stops.pkl"))
validations = pd.read_pickle(os.path.join(INTERIM_DIR,"validations.pkl"))

print({name: df.shape for name, df in
       [("routes", routes), ("vehicles", vehicles), ("stops", stops), ("validations", validations)]})


{'routes': (60, 6), 'vehicles': (180, 5), 'stops': (400, 5), 'validations': (81600, 10)}


In [7]:
#Tracker for dropped rows

reject_frames = []
drop_log = {}

def log_drop(reason: str, subset: pd.DataFrame) -> None:
    if len(subset) == 0:
        return
    tagged = subset.copy()
    tagged["reject_reason"] = reason
    reject_frames.append(tagged)
    drop_log[reason] = drop_log.get(reason, 0) + len(subset)
    print(f"  dropped {len(subset):,} rows -> reason: {reason}")

# Validations

#### 1. Drop exact duplicates and duplicate validation_id

In [8]:
def drop_exact_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.duplicated(keep= "first")
    log_drop("exact_duplicate_row", df[mask])
    return df[~mask].reset_index(drop=True)

def drop_duplicate_validation_id(df: pd.DataFrame) -> pd.DataFrame:
    mask = df.duplicated(subset= "validation_id", keep="first")
    log_drop("duplicate_validation_id", df[mask])
    return df[~mask].reset_index(drop=True)


validations = drop_exact_duplicates(validations)
validations = drop_duplicate_validation_id(validations)
print("remaining rows:", len(validations))



  dropped 802 rows -> reason: exact_duplicate_row
  dropped 798 rows -> reason: duplicate_validation_id
remaining rows: 80000


### 2.Parse validation_ts into real datetime64

In [9]:
parsed_validation_ts = pd.to_datetime(
    validations["validation_ts"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

parsed_validation_ts = parsed_validation_ts.fillna(
    pd.to_datetime(
        validations["validation_ts"], format="%d/%m/%Y %H:%M", errors="coerce"))


parsed_validation_ts = parsed_validation_ts.fillna(
    pd.to_datetime(
        validations["validation_ts"], format="%B %d, %Y %I:%M %p", errors="coerce"
    )
)

unparseable_mask = parsed_validation_ts.isna() & validations["validation_ts"].notna()
log_drop("unparseable_validation_ts", validations[unparseable_mask])


validations["validation_ts"] = parsed_validation_ts

validations = validations[~unparseable_mask].reset_index(drop=True)
print(f"Successfully kept {len(validations):,} clean rows in validations.")

Successfully kept 80,000 clean rows in validations.


### 3.Strip whitespace in all ID_columns:


In [10]:
def strip_whitespace_id(df: pd.DataFrame, id_columns:list) -> pd.DataFrame:
    df = df.copy()
    for col in id_columns:
        df[col] = df [col].str.strip()
        return df

validations = strip_whitespace_id(
    validations,["validation_id", "route_id", "vehicle_id", "stop_id", "card_id"]
)
print("route_id whitespace remaining:", (validations["route_id"].str.strip() !=validations["route_id"]).sum())

route_id whitespace remaining: 11866


### 4. Normalize the fare_type to a fixed set


In [11]:
def normalize_fare_type(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    key = df["fare_type"].str.strip().str.lower()
    df["fare_type"] = key.map(fare_type_mapping).astype("string")
    return df


fare_type_mapping = {
"full fare": "Adult",
"adult": "Adult",
"student": "Student",
"senior":"Senior", 
"senior citizen": "Senior",
"kid": "Child", 
"child": "Child",
"pass holder": "Pass Holder",
"passholder": "Pass Holder", 
"monthly pass": "Pass Holder",}

validations = normalize_fare_type(validations)
unmapped = validations["fare_type"].isna().sum()
print(f"rows with unmapped fare_type after normalization {unmapped}")
print(validations["fare_type"].value_counts(dropna=False))

rows with unmapped fare_type after normalization 0
fare_type
Adult          44401
Student        11847
Senior          9471
Pass Holder     7958
Child           6323
Name: count, dtype: Int64


##### All the original labels mapped clearly to the fixed set with zero unmapped rows.

### 5.Normalize transfer_flag to a proper boolean

In [12]:
transfer_flag_map = {
    "y" : True, "yes" : True, "true" : True, "1" : True,
    "n" : False, "no" : False, "false" : False, "0" : False,
}

key = validations["transfer_flag"].astype(str).str.strip().str.lower()
validations["transfer_flag"] = key.map(transfer_flag_map).astype("boolean")

### 6. Handle passanger_count: nulls.-> drop and log; negatives -> keep but add is_reversed boolean.

In [13]:
def passenger_count_h(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    

    null_mask = df["passenger_count"].isna()
    
    log_drop("null_passenger_count", df[null_mask])
    df = df[~null_mask].reset_index(drop=True)

    df["is_reversed"] = df["passenger_count"] < 0
    
    return df # Added missing return statement

validations = passenger_count_h(validations)
print("remaining rows:", len(validations))
print(validations["is_reversed"].value_counts())



  dropped 811 rows -> reason: null_passenger_count
remaining rows: 79189
is_reversed
False    78831
True       358
Name: count, dtype: int64


### 7. Handle fare_amount : nulls -> impute with median for that fare_type;  
### Values above the 99.5th percentile -> cap at that percentile. Justife the choice in markdown

In [14]:
def impute_fare_amount_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    medians_by_type = df.groupby("fare_type")["fare_amount"].median()
    null_mask = df["fare_amount"].isna()
    df.loc[null_mask, "fare_amount"] = df.loc[null_mask,
"fare_type"].map(medians_by_type)
    print(f"imputed {null_mask.sum():,} null fare_amount values using per-fare_type medians:")
    print(medians_by_type)
    return df


def cap_fare_amount_outliers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    cap = df["fare_amount"].quantile(0.995)
    capped_mask = df["fare_amount"] > cap
    print(f"capping {capped_mask.sum()} rows above the 99.5th percentile (${cap:.2f})")
    df["fare_amount"] = df["fare_amount"].clip(upper=cap)
    return df


validations = impute_fare_amount_nulls(validations)
validations = cap_fare_amount_outliers(validations)
print("\nfare_amount after cleaning:")
print(validations["fare_amount"].describe())

imputed 1,156 null fare_amount values using per-fare_type medians:
fare_type
Adult          2.5
Child          1.0
Pass Holder    0.0
Senior         1.2
Student        1.5
Name: fare_amount, dtype: float64
capping 362 rows above the 99.5th percentile ($2.86)

fare_amount after cleaning:
count    79189.000000
mean         1.835615
std          0.837237
min          0.000000
25%          1.230000
50%          2.310000
75%          2.520000
max          2.860000
Name: fare_amount, dtype: float64


##### It's better to use cap rather than drop in this case because it keeps every validation row in the data set, while preventing outliers from distorting any revenue aggregate, and it only catches the genuine outliers.

### 8.Replace blank card_id with SINGLE_RIDE.

In [15]:
def blank_card_id(df: pd.DataFrame) -> pd.DataFrame : 
    df = df.copy()

    df["card_id"] = df["card_id"].astype(str).str.strip()
    df["card_id"] = df["card_id"].replace(["","nan","None","Nan"],np.nan)

    blank_count = df["card_id"].isna().sum()
    df["card_id"] = df["card_id"].fillna("SINGLE_RIDE")

    print(f"Identified and replaced {blank_count:,} blank card_id entries with 'SINGLE_RIDE'.")

    return df


validations = blank_card_id(validations)
print("SINGLE_RIDE ROWS:", (validations["card_id"] == "SINGLE_RIDE").sum())

Identified and replaced 9,448 blank card_id entries with 'SINGLE_RIDE'.
SINGLE_RIDE ROWS: 9448


# Routes

### 1.Trim and title-case route_name

In [16]:
def clean_route_name(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["route_name"] = df["route_name"].astype(str).str.strip().str.title()
    return df

routes = clean_route_name(routes)

print(routes["route_name"].head())

0       Angel Harbor Line
1     Jeffrey Street Line
2       Erica Plains Line
3    Brittany Bypass Line
4       Clayton Fort Line
Name: route_name, dtype: str


### 2. Standardise mode to Title Case, stripped.

In [17]:
def standardise_mode(df: pd.DataFrame)-> pd.DataFrame:
    df = df.copy()
    df["mode"] = df["mode"].str.strip().str.title()
    return df

routes = standardise_mode(routes)

print(routes["mode"].value_counts())

mode
Bus      34
Tram     20
Metro     6
Name: count, dtype: Int64


### 3. Fill null district with Unknown.  

In [18]:
def full_null_district(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["district"] = df["district"].fillna("Unknown")
    return df

routes = full_null_district(routes)
print(routes[["district"]].isna().sum().to_dict())

{'district': 0}


### 4. Fill null route_lengh_km with the median for that mode.

In [19]:
def null_route_length_km(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    medians_by_mode = df.groupby("mode")["route_length_km"].median()
    null_mask = df["route_length_km"].isna()
    df.loc[null_mask,"route_length_km"] = df.loc[null_mask,"mode"].map(medians_by_mode)
    return df

routes = null_route_length_km(routes)
print(routes[["route_length_km"]].isna().sum().to_dict())

{'route_length_km': 0}


# Vehicles

### 1.Set year_manufactured outside [1980,2026] to null.

In [20]:
def clean_year_manufactured(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    outside_of_range_mask = (df["year_manufactured"] < 1980) | (df["year_manufactured"] > 2026)

    invalid_count = outside_of_range_mask.sum()
    print(f"Identified {invalid_count:,} records with 'year_manufactured'")
    df.loc[outside_of_range_mask,"year_manufactured"] = np.nan
    return df

vehicles = clean_year_manufactured(vehicles)


Identified 15 records with 'year_manufactured'


### 2.Derive vehicle_age_band : 0-5 ,6-10,11-20,20+, Unknown.

In [21]:
REFERENCEE_YEAR = int(validations["validation_ts"].dt.year.max())
print(f"REFERENCE_YEAR for vehicle age banding: {REFERENCEE_YEAR}")

def derive_vehicle_age_band(df: pd.DataFrame ,reference_year:int) -> pd.DataFrame:
    df = df.copy()
    age = reference_year - df["year_manufactured"]
    age = age.clip(lower=0)

    bins = [-1, 5, 10, 20, np.inf]
    labels= ["0-5", "6-10", "11-20", "20+"]
    band = pd.cut(age,bins=bins, labels=labels).astype("string")
    band = band.where(df["year_manufactured"].notna(), "Unknown")
    df["vehicle_age_band"] = band.fillna("Unknown")
    return df



vehicles = derive_vehicle_age_band(vehicles, REFERENCEE_YEAR)
print(vehicles["vehicle_age_band"].value_counts(dropna=False))

REFERENCE_YEAR for vehicle age banding: 2024
vehicle_age_band
20+        77
11-20      43
6-10       23
0-5        22
Unknown    15
Name: count, dtype: Int64


### 3.Fill null capacity with the median for that vehicle_type.

In [22]:
def fill_null_capacity(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    medians_by_type = df.groupby("vehicle_type")["capacity"].median()
    null_mask = df["capacity"].isna()
    df.loc[null_mask, "capacity"] = df.loc[null_mask,"vehicle_type"].map(medians_by_type).round().astype("Int64")
    print(f"imputed {null_mask.sum()} null capacity values using per-vehicle_type medians:")
    print(medians_by_type)
    return df


vehicles = fill_null_capacity(vehicles)
print("\nnulls remaining:", vehicles[["capacity","year_manufactured"]].isna().sum().to_dict())

imputed 12 null capacity values using per-vehicle_type medians:
vehicle_type
Articulated Bus    118.0
Metro Set          619.0
Standard Bus        69.0
Tram               201.0
Name: capacity, dtype: Float64

nulls remaining: {'capacity': 0, 'year_manufactured': 15}


# Stops

### 1.Treim and title-case stop_name.

In [23]:
def clean_stop_name(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["stop_name"] = df["stop_name"].str.strip().str.title()
    return df

stops = clean_stop_name(stops)
print(stops["stop_name"].head())

0    Matthew Radial Stop
1    Brianna Avenue Stop
2       Jean Circle Stop
3     Carlos Bypass Stop
4     Jones Freeway Stop
Name: stop_name, dtype: string


### 2. Nprmalize has_shelter to boolean.

In [24]:

def normalized_has_shelter(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Clean the strings (lowercase, remove spaces)
    cleaned_series = df["has_shelter"].astype(str).str.strip().str.lower()

    # 2. Define what counts as True and what counts as Missing/NaN
    true_values = ["yes", "true", "1", "1.0"]
    null_values = ["nan", "none", "", "nat"]

    # 3. FIXED: Cast the column to the nullable "boolean" dtype first
    # This allows the column to store True, False, and missing states together
    df["has_shelter"] = cleaned_series.isin(true_values).astype("boolean")

    # 4. Safely apply native missing tokens without throwing a TypeError
    df.loc[cleaned_series.isin(null_values), "has_shelter"] = pd.NA

    return df


# Run the updated function
stops = normalized_has_shelter(stops)

# Display results
print(stops["has_shelter"].value_counts(dropna=False))
print("\nFirst 5 stop names:")
print(stops["stop_name"].head())


has_shelter
False    313
True      87
Name: count, dtype: Int64

First 5 stop names:
0    Matthew Radial Stop
1    Brianna Avenue Stop
2       Jean Circle Stop
3     Carlos Bypass Stop
4     Jones Freeway Stop
Name: stop_name, dtype: string


# Referential integrity

### 1. Report and drop validations whose route_id, vehicle_id, or stop_id doesn't exist in the master files.Validations with card_id= Single_Ride are valid and must be kept

In [25]:
def validations_integrity( df, routes_df, vehicles_df, stops_df) -> pd.DataFrame:
    df = df.copy()
    valid_routes = set(routes_df["route_id"])
    valid_vehicles = set(vehicles_df["vehicle_id"])
    valid_stops = set(stops_df["stop_id"])

    bad_route = ~df["route_id"].isin(valid_routes)
    bad_vehicles = ~df["vehicle_id"].isin(valid_vehicles)
    bad_stop = ~df["stop_id"].isin(valid_stops)
    any_bad = bad_route | bad_vehicles | bad_stop

    reasons = np.select(
        [bad_route & bad_vehicles & bad_stop,
         bad_route & bad_vehicles,
         bad_route & bad_stop,
         bad_vehicles & bad_stop,
         bad_stop,
         bad_vehicles,
         bad_route],
         ["o_route_vehicle_stop","o_route_vehicle","o_route_stop","o_vehicle_stop","o_stop_id","o_vehicle_id","o_route_id"],
         default="",
    )

    for reason in np.unique(reasons[any_bad]):
        mask = any_bad & (reasons == reason)
        log_drop(reason,df[mask])

    single_ride_dropped = ((df["card_id"] == "SINGLE_RIDE") & any_bad).sum()
    single_ride_total = (df["card_id"] == "SINGLE_RIDE").sum()
    print(f"SINGLE_RIDE rows dropped: {single_ride_dropped} of {single_ride_total}"
          f"(Dropped for a bad route/vehicle/stop id-"
          f"because card_id as itself was never the reason)")

    return df[~any_bad].reset_index(drop=True)


validations = validations_integrity(validations,routes,vehicles,stops)
print("Remaining rows:", len(validations))

  dropped 11,756 rows -> reason: o_route_id
  dropped 5 rows -> reason: o_route_stop
  dropped 2 rows -> reason: o_route_vehicle
  dropped 8 rows -> reason: o_stop_id
  dropped 11 rows -> reason: o_vehicle_id
SINGLE_RIDE rows dropped: 1396 of 9448(Dropped for a bad route/vehicle/stop id-because card_id as itself was never the reason)
Remaining rows: 67407


# Derived columns

### On validations

In [27]:
DAY_NAMES_EN = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]



def derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df=df.copy()
    ts = df["validation_ts"]

    df["date"] = ts.dt.date
    df["year"] = ts.dt.year
    df["month"] = ts.dt.month
    df["year_month"] = ts.dt.strftime("%Y-%m")
    df["quarter"] = ts.dt.quarter
    df["hour"] = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek.map(dict(enumerate(DAY_NAMES_EN)))
    df["is_weekend"] = ts.dt.dayofweek.isin([5,6])

    conditions = [
        df["hour"].between(4, 6),
        df["hour"].between(7, 9),
        df["hour"].between(10, 15),
        df["hour"].between(16, 18),
        df["hour"].between(19, 23),
    ]

    choices = ["Early (04-06)", 
               "AM peak (07-09)", 
               "Midday (10-15)", 
               "PM Peak(16-18)", 
               "Evening(19-23)"]
    df["time_period"] = np.select(conditions, choices, default="Night(00-03)")

    df["revenue"] = df["fare_amount"] * df["passenger_count"]
    return df


validations = derived_columns(validations)
print(validations[["validation_ts", "date", "year_month","quarter","hour","day_of_week","is_weekend","time_period","revenue"]].head())


        validation_ts        date year_month  quarter  hour day_of_week  is_weekend      time_period  revenue
0 2024-06-06 08:48:00  2024-06-06    2024-06        2     8    Thursday       False  AM peak (07-09)     2.48
1 2024-03-06 13:15:00  2024-03-06    2024-03        1    13   Wednesday       False   Midday (10-15)     2.85
2 2024-06-14 18:35:27  2024-06-14    2024-06        2    18      Friday       False   PM Peak(16-18)     1.34
3 2024-02-16 18:30:59  2024-02-16    2024-02        1    18      Friday       False   PM Peak(16-18)     2.47
4 2024-03-03 08:41:00  2024-03-03    2024-03        1     8      Sunday        True  AM peak (07-09)     0.13


# Save cleaned frames and rejects

In [30]:
for name, df in [("routes", routes), ("vehicles",vehicles), ("stops",stops), ("validations",validations)]:
    out_path = os.path.join(INTERIM_DIR, f"{name}_clean.pkl")
    df.to_pickle(out_path)
    print(f"saved{out_path} ({len(df):,}rows)")

if reject_frames:
    rejects_df = pd.concat(reject_frames, ignore_index=True)
    rejects_path =os.path.join(REJECTED_DIR, "validation_rejects.csv")
    rejects_df.to_csv(rejects_path, index=False)
    print(f"\nsaved {rejects_path} ({len(rejects_df):,} rows)")
else:
    print("\nno rejected rows to save")

saved..\data\interim\routes_clean.pkl (60rows)
saved..\data\interim\vehicles_clean.pkl (180rows)
saved..\data\interim\stops_clean.pkl (400rows)
saved..\data\interim\validations_clean.pkl (67,407rows)

saved ..\data\rejects\validation_rejects.csv (14,193 rows)


# Summary (droped rows and reason):

In [32]:
summary = pd.DataFrame(
    [{"reason": reason, "rows_dropped" : count} for reason, count in drop_log.items()]
).sort_values("rows_dropped",ascending=False)

total_dropped = summary["rows_dropped"].sum()
print(summary.to_string(index=False))
print(f"\ntotal rows that are droped : {total_dropped:,}")
print(f"validations row count: {len(validations):,}")

                 reason  rows_dropped
             o_route_id         11756
   null_passenger_count           811
    exact_duplicate_row           802
duplicate_validation_id           798
           o_vehicle_id            11
              o_stop_id             8
           o_route_stop             5
        o_route_vehicle             2

total rows that are droped : 14,193
validations row count: 67,407
